In [2]:
import numpy as np
import torch
from kan import KAN

# import config as cfg


import sys
sys.path.append('../')

In [7]:
from train import make_model, build_kan_width # testing

In [6]:
# copied from train.py
def make_model(kan_width, seed, device):
    torch.manual_seed(seed)
    m = KAN(width=kan_width, grid=5, k=3, seed=seed, device=device, auto_save=False, grid_eps=0.0)
    m.speed()
    return m

In [9]:
width = build_kan_width(16)
width # nice

[16, 16, 16, 16, 16]

In [10]:
model = make_model(width, seed=42, device='cpu')

In [ ]:
model # we got a model

MultKAN(
  (act_fun): ModuleList(
    (0-3): 4 x KANLayer(
      (base_fun): SiLU()
    )
  )
  (base_fun): SiLU()
  (symbolic_fun): ModuleList(
    (0-3): 4 x Symbolic_KANLayer()
  )
)

In [41]:
num_layers = model.depth
last_layer = model.depth - 1
print(last_layer) # cool 

model.act_fun[last_layer].in_dim, model.act_fun[last_layer].out_dim

3


(16, 16)

# OK. we need to tweak our KAN constructor so that we get the in and out dim of the final layer to always be (num_rxns, num_species). In our case, 13 and 16

In [42]:
def build_kan_width2(n_species, num_fluxes): # tweak your existing function
    return [n_species, 16, 16, num_fluxes, n_species]

In [ ]:
width = build_kan_width2(16, 13)
width # great

[16, 16, 16, 13, 16]

In [46]:
# recreate a model
model = make_model(width, 42, 'cpu')
model

MultKAN(
  (act_fun): ModuleList(
    (0-3): 4 x KANLayer(
      (base_fun): SiLU()
    )
  )
  (base_fun): SiLU()
  (symbolic_fun): ModuleList(
    (0-3): 4 x Symbolic_KANLayer()
  )
)

In [48]:
last = model.depth-1
model.act_fun[last].in_dim, model.act_fun[last].out_dim # cool

(13, 16)

# One idea (possibly too brute force)

We can create the stoichiometry matrix, and read off that to create our own functions. Below is the stoich matrix

In [52]:
# stoichiometry matrix has dimensions (num_species, num_rxns)
# this could stand to be double checked
S = np.array([[1, -1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], # O3
              [1, -1, 0, 0, 0, -1, 0, 0, 0, 0, 0, 0, 0], # NO
              [-1, 1, 0, 0, 0, 1, -1, 0, 0, 0, 0, -1, 1], # NO2
              [0, 0, -1, -1, -1, 0, 0, 0, 0, 0, 0, 0, 0], # HCHO
              [0, 0, 2, 0, 1, -1, 0, 0, 1, 0, 1, 0, 0], # HO2
              [0, 0, 0, 0, 0, 0, 0, -1, -1, 0, 0, 0, 0], # H2O2
              [0, 0, 0, 0, -1, 1, -1, 2, -1, -1, 0, 0, 0], # OH
              [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0], # HNO3
              [0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0], # CO
              [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], # H2
              [0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0, 0], # ALD2
              [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0], # MGLY
              [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, -1, 1], # MCO3
              [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, -1], # PAN
              [0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0], # H2O
              [-1, 1, -2, 0, 0, 0, 0, 0, 0, -1, -2, 0, 0]]) # O2
S.shape

(16, 13)

## And here one way I figured out we can fix activations according to whatever function we want

The part that does it is the `lambda x: S_row[j_idx] * x` argument in `.fix_symbolic()`. 
The loop starting at Line 4 goes through 

1) Every node in the L-1 layer of the KAN (here, our 13 "reaction flux" nodes), 
2) Assigns scaled linear activation functions to each activation emanating from this reaction flux (16 of these, many of which are 0 as you can see from S above). This corresponds to the stoichiometry of each species participating in that reaction. Each activation points to a species tendency output node.

After this, activation function outputs are aggregated (summed) for each node. Since many activations are 0, this should be no problem.

In [61]:
last_layer = model.act_fun[last]
LAST = model.depth-1

# loop through final act functions
for i_idx in range(last_layer.in_dim):
    S_row = S[:,i_idx]
    for j_idx in range(last_layer.out_dim):
        model.fix_symbolic(LAST, i_idx, j_idx, lambda x: S_row[j_idx] * x, fit_params_bool=False)

# freeze the whole layer
for p in model.act_fun[LAST].parameters():
    p.requires_grad_(False)
for p in model.symbolic_fun[LAST].parameters():
    p.requires_grad_(False)

print(model.act_fun[LAST].mask)                    # should be all 0 (splines off)
print(model.symbolic_fun[LAST].mask)               # should be all 1 (symbolic on)
print([p.requires_grad for p in model.act_fun[LAST].parameters()])

Parameter containing:
tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])
Parameter conta

This above code chunk can exist in your `make_model()` function. You can also create a class that has this code chunk as a function, e.g. 

In [70]:
class KANBuilder(torch.nn.Module):
    def __init__(self, kan_width, seed, device):
        super().__init__()

        self.kan_width = kan_width
        self.seed = seed
        self.device = device
        self.model = self.make_model()

    def make_model(self):
        torch.manual_seed(self.seed)
        m = KAN(width=self.kan_width, grid=5, k=3, seed=self.seed, device=self.device, auto_save=False, grid_eps=0.0)
        m.speed()
        return m

    def fix_activations(self, ...):
        ## Fill in here
        # works on self.model
        # does not have to return anything
        # should take stoich matrix as arg
    

SyntaxError: invalid syntax (1754638340.py, line 16)

This might be overkill, and adds an annoying extra piece of code to manage, i.e.

In [ ]:
smog_kan = KANBuilder()
smog_kan.model
...

But you get the idea

## Caveats:

1) according to Claude (lol), updating grid will reset the requires_grad argument to True. SO, after every grid update, we'd have to reset the requires_grad for every function in the last layer. 
2) this could be fine especially if we prune prior to the first grid update. that is, all those 0 functions could be taken out, and an exhaustive loop to reset requires_grad/freeze functions wouldn't take as much compute (though negligible either way)

## Short-term solution (for beta-testing)
1) Can you try training one version of a model with fixed final layer activations with setting update_grid=False within `model.fit()`?